# ARC-AGI-3 null-agent walkthrough

This notebook explains the first architecture before we add genetic programming, grammatical optimization, learned world models, or MCTS.

The rule for this project is simple: **the notebook demonstrates shared source code; it does not contain a separate solver.**

Conceptual path:

`frame/history → observation helpers → policy → official environment → scorecard`

Future path:

`perception → evidence → grammar hypotheses → inferred world model → internal MCTS → real action`

## 1. Load the shared project code

Run the notebook from the repository root with the project's Python 3.12 environment.

```bash
make setup
```

The imports below are the same modules used by `agent/my_agent.py`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from arc_fun.observations import first_grid, summarize_frame
from arc_fun.policy import NullPolicy

## 2. Why start with a null policy?

A null baseline should be boring, deterministic, and incapable of exploiting a specific public game.

Its job is to prove that a candidate policy can move through the exact same interface that later grammar-search candidates will use.

For a running game it selects the first available non-reset action. For a complex coordinate action it chooses the visible grid center. It resets only when the game has not started or has ended in `GAME_OVER`.

In [ ]:
policy = NullPolicy()
print(policy.name)

## 3. Run the real local ARC environment

The easiest competition-faithful loop is the repository script:

```bash
make list-games
make play-local GAME=ls20 STEPS=80
```

That script creates the official local `Arcade`, wraps the shared policy in `MyAgent`, runs the official framework loop, and prints the ARC scorecard.

This notebook intentionally does not duplicate that loop.

## 4. Where structural regression becomes program induction

In the earlier structural-regression setup, search produced mathematical structures and rewarded them by downstream predictive quality.

Here a candidate can instead represent a transition hypothesis:

`T_hat(S_t, A_t) -> S_hat_(t+1)`

Evidence arrives as real transitions `(S_t, A_t, S_(t+1))`.

The grammar can generate candidate executable hypotheses, then score them by consistency with observed transitions plus a complexity penalty.

## 5. Where stochastic MCTS belongs

On hidden competition games, the real ARC environment should **not** be treated as a rollout oracle.

```text
observed transitions
      ↓
candidate grammar programs
      ↓
rank / update hypotheses
      ↓
internal transition model
      ↓
MCTS over hypothetical futures
      ↓
select one real action
      ↓
new evidence
```

That keeps real actions scarce while letting internal computation be large.

## 6. Fitness loop for future GP

Once a genome can compile into a policy, the outer experiment becomes conceptually:

```python
for candidate in population:
    policy = compile_candidate(candidate)
    results = evaluate_fixed_game_seed_panel(policy)
    candidate.fitness = aggregate(results) - complexity_penalty(candidate)
```

Critical validation rule: hold out **entire games**, not merely seeds. Otherwise evolution can memorize mechanics of the public training environments.

## 7. End-to-end deployment path

```text
shared policy code
   ├─ local: make play-local
   └─ Kaggle: make notebook → make submit
                              ↓
                      Phase A notebook run
                              ↓
                    Submit to Competition
                              ↓
                       hidden game rerun
                              ↓
                        leaderboard score
```

The generated Kaggle notebook embeds the same `src/arc_fun` package and the same thin `MyAgent` adapter used locally.